### Task 5 - Model Iteration - Logistic Regression


In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
)
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import RandomOverSampler

Loading the Dataset

In [2]:
df = pd.read_csv("final_filtered_dataset.csv", encoding="ISO-8859-1")

d:\anaconda\envs\block_c_y2\lib\site-packages\IPython\core\interactiveshell.py:3508: DtypeWarning: Columns (8,17,18,19,20,21,22,23,24,25) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


Cleaning the Data

In [ ]:
df = df.dropna(subset=["Corrected_Emotion"])

Cleaning the text

In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)  # remove punctuation
    text = re.sub(r"\s+", " ", text)  # remove extra whitespace
    return text.strip()


df["Cleaned_Sentence"] = df["Corrected Sentence"].apply(clean_text)

Removing duplicate rows

In [5]:
df = df.drop_duplicates(subset=["Cleaned_Sentence", "Corrected_Emotion"])

Encoding the Labels

In [6]:
le = LabelEncoder()
df["Emotion_Label"] = le.fit_transform(df["Corrected_Emotion"])

# Optional: See label mappings
print(dict(zip(le.classes_, le.transform(le.classes_))))

{'anger': 0, 'disgust': 1, 'fear': 2, 'happiness': 3, 'neutral': 4, 'sadness': 5, 'surprise': 6}


Preparing the Features

In [7]:
X = df["Cleaned_Sentence"]
y = df["Emotion_Label"]

Vectorizing the Text

In [8]:
vectorizer = TfidfVectorizer(max_features=5000)
X_vectorized = vectorizer.fit_transform(X)

Balancing the Classes (Oversample)

In [9]:
ros = RandomOverSampler(random_state=42)
X_balanced, y_balanced = ros.fit_resample(X_vectorized, y)

Splitting into train and test sets

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, test_size=0.2, stratify=y_balanced, random_state=42
)

Training Logistic Regression

In [11]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

Evaluation of the model

In [12]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 0.884175580966433
              precision    recall  f1-score   support

       anger       0.94      0.97      0.96       387
     disgust       0.96      1.00      0.98       387
        fear       0.87      0.95      0.90       387
   happiness       0.87      0.81      0.84       388
     neutral       0.80      0.60      0.69       387
     sadness       0.88      0.95      0.92       387
    surprise       0.85      0.91      0.88       388

    accuracy                           0.88      2711
   macro avg       0.88      0.88      0.88      2711
weighted avg       0.88      0.88      0.88      2711

